# FIT Profile Preparation and LSTM Trial

We read the 56 measured axle FIT files, prepare 112 wheel-coordinate sequences
and try an LSTM using an any-rail displacement-hunting label. The two wheels
of an axle share that label. Follow [INSTALL.md](../INSTALL.md) for setup.

This prototype splits wheels randomly. Related wheels can be in both sets,
and the later augmentation happens before another random split. Its native
accuracy is not an estimate for unseen cars and is not part of the common
truck/rail model ranking.

In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src" / "project_paths.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the repository; see INSTALL.md.")
sys.path.insert(0, str(ROOT / "src"))
from project_paths import data_directory, find_project_root, required_file
ROOT = find_project_root(ROOT)
DATA = data_directory(ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import random

import tensorflow as tf
from tensorflow.keras import layers, models

In [ ]:
axle_list = np.array(["Car1_Axle1.fit", "Car1_Axle2.fit", "Car1_Axle3.fit", "Car1_Axle4.fit", "Car2_Axle1.fit", "Car2_Axle2.fit", "Car2_Axle3.fit", "Car2_Axle4.fit", "Car3_Axle1.fit", "Car3_Axle2.fit", "Car3_Axle3.fit", "Car3_Axle4.fit", "Car4_Axle1.fit", "Car4_Axle2.fit", "Car4_Axle3.fit", "Car4_Axle4.fit", "Car5_Axle1.fit", "Car5_Axle2.fit", "Car5_Axle3.fit", "Car5_Axle4.fit", "Car6_Axle1.fit", "Car6_Axle2.fit", "Car6_Axle3.fit", "Car6_Axle4.fit", "Car7_Axle1.fit", "Car7_Axle2.fit", "Car7_Axle3.fit", "Car7_Axle4.fit", "Car8_Axle1.fit", "Car8_Axle2.fit", "Car8_Axle3.fit", "Car8_Axle4.fit", "Car9_Axle1.fit", "Car9_Axle2.fit", "Car9_Axle3.fit", "Car9_Axle4.fit", "Car10_Axle1.fit", "Car10_Axle2.fit", "Car10_Axle3.fit", "Car10_Axle4.fit", "Car11_Axle1.fit", "Car11_Axle2.fit", "Car11_Axle3.fit", "Car11_Axle4.fit", "Car12_Axle1.fit", "Car12_Axle2.fit", "Car12_Axle3.fit", "Car12_Axle4.fit", "Car13_Axle1.fit", "Car13_Axle2.fit", "Car13_Axle3.fit", "Car13_Axle4.fit", "Car14_Axle1.fit", "Car14_Axle2.fit", "Car14_Axle3.fit", "Car14_Axle4.fit"])
axle_count = len(axle_list)


In [ ]:
# Read the two wheel profiles from each FIT file.
rxs = []
rys = []
lxs = []
lys = []

for axle in axle_list:
    df = pd.read_csv(required_file(DATA / "Profiles" / "Candidate wheel profiles" / axle), skiprows = 3)
    cap = len(df)
    xstep = []
    ystep = []
    side = "right"
    
    for i in range(cap - 2):
        step = df.iloc[i].str.split().tolist()[0]
        
        if len(step) == 1:
            side = "left" 
            rxs.append(xstep)
            rys.append(ystep)
            xstep = []
            ystep = []
        elif (side == "left"):
            xstep.append(float(step[0]))
            ystep.append(float(step[1]))
        elif (side == "right"):
            xstep.append(float(step[0]))
            ystep.append(float(step[1]))

    lxs.append(xstep)
    lys.append(ystep)



In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(6, 6))

for i in range(axle_count):
    ax[0].scatter(rxs[i], rys[i])
    ax[0].set_title("Right Wheel")
    
    ax[1].scatter(lxs[i], lys[i])
    ax[1].set_title("Left Wheel")

for axis in ax:
    axis.set_xlabel("X (file coordinates)")
    axis.set_ylabel("Y (file coordinates)")
fig.suptitle("Original FIT coordinates")
plt.tight_layout()
plt.show()

In [ ]:
# Give both wheels the axle label: hunting on any of the seven rails.
fit_res = pd.read_excel(required_file(DATA / "Eq_Con_Fit_Results.xlsm"))

hunt_df = fit_res[["Filename", "Empty Axle from Lat Disp HS [mph]", "Loaded Axle from Lat Disp HS [mph]"]]
hunt_df = hunt_df[~hunt_df['Filename'].isin(["KRWHL_RProfile 1.wnt", "KRWHL_RProfile 2.wnt", "KRWHL_RProfile 3.wnt", "KRWHL_RProfile 4.wnt", "KRWHL_RProfile 5.wnt", "KRWHL_RProfile 6.wnt","KRWHL_RProfile 7.wnt"])]
hunt_df.reset_index(drop=True, inplace=True)

hunting_empty = []
hunting_full = []
ehunt = False
fhunt = False
step = 0

for i in range(len(hunt_df["Filename"])):
    if hunt_df["Empty Axle from Lat Disp HS [mph]"][i] != 80:
        ehunt = True

    if hunt_df["Loaded Axle from Lat Disp HS [mph]"][i] != 80:
        fhunt = True

    if step == 6:
        step = -1
        if ehunt == True:
            hunting_empty.append(True)
            ehunt = False
        else: 
            hunting_empty.append(False)

        if fhunt == True:
            hunting_full.append(True)
            fhunt = False
        else: 
            hunting_full.append(False)
    
    step += 1

y_empty_temp = [hunting_empty, hunting_empty]
y_empty = [item for sublist in y_empty_temp for item in sublist]
y_empty = np.array(y_empty)

In [ ]:
# Use 28 points per wheel so the sequences have the same length.

# Trim three longer profiles.
ils = [27, 28, 49]
rxs[ils[0]], rys[ils[0]] = rxs[ils[0]][2:30], rys[ils[0]][2:30]

endstep = len(rxs[ils[1]])
stepxs = rxs[ils[1]][0:endstep - 11] + rxs[ils[1]][endstep - 10:endstep - 8] + rxs[ils[1]][endstep-7:endstep-6] + rxs[ils[1]][endstep-5:endstep]
stepys = rys[ils[1]][0:endstep - 11] + rys[ils[1]][endstep - 10:endstep - 8] + rys[ils[1]][endstep-7:endstep-6] + rys[ils[1]][endstep-5:endstep]
rxs[ils[1]], rys[ils[1]] = stepxs, stepys

endstep2 = len(rxs[ils[2]])
stepxs2 = rxs[ils[2]][0:endstep2 - 7] + rxs[ils[2]][endstep2 - 6:endstep2]
stepys2 = rys[ils[2]][0:endstep2 - 7] + rys[ils[2]][endstep2 - 6:endstep2]
rxs[ils[2]], rys[ils[2]] = stepxs2, stepys2

# Remove selected outlier points.
rxs[40], rys[40] = rxs[40][1:len(rxs[40])-1], rys[40][1:len(rys[40])-1]
rxs[41], rys[41] = rxs[41][1:len(rxs[41])-1], rys[41][1:len(rys[41])-1]
rxs[42], rys[42] = rxs[42][1:len(rxs[42])-1], rys[42][1:len(rys[42])-1]
rxs[43], rys[43] = rxs[43][1:len(rxs[43])-1], rys[43][1:len(rys[43])-1]

lxs[40], lys[40] = lxs[40][1:len(lxs[40])-1], lys[40][1:len(lys[40])-1]
lxs[41], lys[41] = lxs[41][1:len(lxs[41])-1], lys[41][1:len(lys[41])-1]
lxs[42], lys[42] = lxs[42][1:len(lxs[42])-1], lys[42][1:len(lys[42])-1]
lxs[43], lys[43] = lxs[43][1:len(lxs[43])-1], lys[43][1:len(lys[43])-1]

# Interpolate profiles with fewer than 28 points.

for i in range(len(rxs)):
    rs = rxs[i]
    rsy = rys[i]
    ls = lxs[i]
    lsy = lys[i]
    
    while len(rs) != 28:
        whle = len(rs)
        frs, brs = rs[0:15], rs[15:whle]
        frsy, brsy = rsy[0:15], rsy[15:whle]
        rs = frs + [(frs[len(frs)-1] + brs[0])/2] + brs
        rsy = frsy + [(frsy[len(frsy)-1] + brsy[0])/2] + brsy
    rxs[i] = rs
    rys[i] = rsy
    
    while len(ls) != 28:
        whle = len(ls)
        fls, bls = ls[0:15], ls[15:whle]
        flsy, blsy = lsy[0:15], lsy[15:whle]
        ls = fls + [(fls[len(fls)-1] + bls[0])/2] + bls
        lsy = flsy + [(flsy[len(flsy)-1] + blsy[0])/2] + blsy
    lxs[i] = ls
    lys[i] = lsy



In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(6, 6))

for i in range(axle_count):
    ax[0].scatter(rxs[i], rys[i])
    ax[0].set_title("Right Wheel")
    
    ax[1].scatter(lxs[i], lys[i])
    ax[1].set_title("Left Wheel")

for axis in ax:
    axis.set_xlabel("X (file coordinates)")
    axis.set_ylabel("Y (file coordinates)")
fig.suptitle("Prepared FIT coordinates")
plt.tight_layout()
plt.show()

In [ ]:
xr = []
xl = []
wheel = []

for i in range(axle_count):
    for j in range(len(rxs[i])):
        wheel.append([rxs[i][j], rys[i][j]])
    xr.append(wheel)
    wheel = []

for i in range(axle_count):
    for j in range(len(lxs[i])):
        wheel.append([lxs[i][j], lys[i][j]])
    xl.append(wheel)
    wheel = []

x = np.array(xr + xl)
np.array(x).shape

# 112 wheels with 28 (x, y) points each.

In [ ]:
xe_train, xe_test, ye_train, ye_test = train_test_split(x, y_empty, test_size = 30, random_state = 67)

In [ ]:
model = models.Sequential([
        layers.Input(shape=(28,2)),
        layers.LSTM(90, return_sequences=False),
        layers.Dropout(0.3),
        layers.Dense(28, activation="relu"),
        layers.Dense(1, activation="sigmoid"),])

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy", tf.keras.metrics.AUC(name="auc")],)

model.fit(xe_train, ye_train, epochs = 100, batch_size = len(xe_train), validation_split = 0.2,)

In [ ]:
pred = (model.predict(xe_test) > 0.5).T[0]

print("Accuracy found to be at around" , 100*int(sum(ye_test == pred))/len(pred), "Percent")

## Before augmentation

Compare test accuracy with the share of wheels labeled as hunting. A model
that predicts hunting for every wheel would match that share without
distinguishing the two classes. The data split is fixed, but training is
unseeded and predictions can vary between runs.

In [ ]:
# Make shifted copies of existing wheel profiles.


def new_wheel_func(x, old_wheel_num):
    new_wheel = []
    new_x = random.uniform(0.001, 0.1)
    new_y = random.uniform(0.001, 0.1)
    for i in range(28):
        xa, ya = x[old_wheel_num][i]
        new_wheel.append([new_x + xa,new_y + ya])

    return new_wheel


In [ ]:
fal_ind = [i for i, val in enumerate(y_empty) if not val]
ls_chosen = fal_ind + fal_ind + fal_ind[0:18] + [3] + [6]
ls_tf = []

for index in ls_chosen:
    ls_tf.append(bool(y_empty[index]))    


for i in range(len(ls_tf)):
    new_wheel = new_wheel_func(x, ls_chosen[i])
    x = np.append(x, [new_wheel], axis = 0)
    y_empty = np.append(y_empty, ls_tf[i])


In [ ]:
plt.figure(figsize=(7, 5))
for i in range(len(x)):
    plt.scatter(x[i][:].T[0], x[i][:].T[1])
plt.xlabel("X (file coordinates)")
plt.ylabel("Y (file coordinates)")
plt.title("Original and translated FIT sequences")
plt.tight_layout()
plt.show()


In [ ]:
xe_train, xe_test, ye_train, ye_test = train_test_split(x, y_empty, test_size = 30, random_state = 67)

model = models.Sequential([
        layers.Input(shape=(28,2)),
        layers.LSTM(90, return_sequences=False),
        layers.Dropout(0.3),
        layers.Dense(28, activation="relu"),
        layers.Dense(1, activation="sigmoid"),])

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy", tf.keras.metrics.AUC(name="auc")],)

model.fit(xe_train, ye_train, epochs = 100, batch_size = len(xe_train), validation_split = 0.2,)

In [ ]:
pred = (model.predict(xe_test) > 0.5).T[0]
print("Accuracy found to be at around" , 100*int(sum(ye_test == pred))/len(pred), "Percent")

## After augmentation

The new sequences are shifted copies of measured wheels. The second split can
put a copy and its original on opposite sides. It also changes the test
sample, so the two scores cannot isolate the effect of augmentation. To
evaluate on unseen cars, split by car before creating copies.